# 03b · Modelos Avanzados — Dengue Con Signos de Alarma (DCSA)
**Fase:** Modelado predictivo — portafolio de 7 algoritmos

**Portafolio:** Logistic Regression, Random Forest, Extra Trees,
Gradient Boosting, XGBoost, LightGBM, CatBoost

**Diseno experimental:**
- Particion 80/20 estratificada
- Validacion 5-fold CV estratificada sobre train
- SMOTE aplicado solo en train (nunca en test)
- Metrica primaria: F1-score (clase positiva = hospitalizado)

> **Por que SMOTE?** SMOTE genera ejemplos sinteticos interpolando entre vecinos
> reales. Se aplica solo en entrenamiento para evitar data leakage.

In [ ]:
import sys
sys.path.insert(0,'../../../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (f1_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score, average_precision_score,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
import config
from config import (DATA_INTERIM,DATA_PROC,OUT_FIGURES,OUT_TABLES,
    CHECKPOINTS,OUT_MODELS,COLORS,RANDOM_STATE,TEST_SIZE,CV_FOLDS,TARGET_COL,
    SIGNOS_ALARMA,SINTOMAS_GENERALES)
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi':150,'savefig.dpi':300,
    'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.titlesize':13,'axes.titleweight':'bold','legend.frameon':False})
df_con = joblib.load(DATA_INTERIM/'datos_con_signos.pkl')
print(f'DCSA: {df_con.shape[0]:,} | Hosp: {df_con[TARGET_COL].mean():.1%}')

In [ ]:
# Seleccion de features y particion
# Se EXCLUYE CLASFINAL para evitar data leakage:
# CLASFINAL domina la prediccion y oculta la contribucion de sintomas
vars_clinicas  = SINTOMAS_GENERALES + SIGNOS_ALARMA
vars_demo      = ['SEXO_','DEF_CLAS_EDAD','TIP_SS_','ESTRATO_','GP_GESTAN']
vars_temporales = ['AÑO','SEMANA']
FEATURES = vars_clinicas + vars_demo + vars_temporales

df_m = df_con[FEATURES+[TARGET_COL]].copy()
for v in vars_clinicas:
    df_m[v] = df_m[v].map({1:1, 2:0})
df_m = df_m.dropna(subset=[TARGET_COL])

X = df_m[FEATURES]
y = df_m[TARGET_COL].astype(int)

X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Prevalencia train: {y_train.mean():.1%} | test: {y_test.mean():.1%}')

joblib.dump((X_train,X_test,y_train,y_test,FEATURES), DATA_PROC/'particion_DCSA.pkl')
print('Particion guardada')

In [ ]:
# Preprocesador
# Numericas: mediana + estandarizacion
# Binarias: moda (no requieren escala)
vars_num = vars_demo + vars_temporales
vars_bin = vars_clinicas

preprocesador = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), vars_num),
    ('bin', SimpleImputer(strategy='most_frequent'), vars_bin),
], remainder='drop')

def make_pipe(clf):
    return ImbPipeline([
        ('prep', preprocesador),
        ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ('clf', clf),
    ])

portafolio = {
    'Logistic Regression': make_pipe(
        LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)),
    'Random Forest': make_pipe(
        RandomForestClassifier(n_estimators=300, max_depth=10,
            random_state=RANDOM_STATE, n_jobs=-1)),
    'Extra Trees': make_pipe(
        ExtraTreesClassifier(n_estimators=300, max_depth=10,
            random_state=RANDOM_STATE, n_jobs=-1)),
    'Gradient Boosting': make_pipe(
        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
            max_depth=4, random_state=RANDOM_STATE)),
    'XGBoost': make_pipe(
        XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
            eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)),
    'LightGBM': make_pipe(
        LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
            num_leaves=31, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)),
    'CatBoost': make_pipe(
        CatBoostClassifier(iterations=300, learning_rate=0.05,
            depth=6, random_seed=RANDOM_STATE, verbose=0)),
}
print(f'Portafolio: {len(portafolio)} modelos')

In [ ]:
# Validacion cruzada 5-fold estratificada
# Se evalua CV ANTES de ver el test set para evitar ajuste al test
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'f1':                make_scorer(f1_score, pos_label=1),
    'roc_auc':           make_scorer(roc_auc_score, needs_proba=True),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'precision':         make_scorer(precision_score, pos_label=1, zero_division=0),
    'recall':            make_scorer(recall_score, pos_label=1),
    'average_precision': make_scorer(average_precision_score, needs_proba=True),
}
resultados_cv = {}
for nombre, pipe in portafolio.items():
    print(f'Entrenando {nombre}...', end=' ')
    cv_res = cross_validate(pipe, X_train, y_train,
        cv=cv, scoring=scoring, n_jobs=-1)
    resultados_cv[nombre] = {m:{'mean':cv_res[f'test_{m}'].mean(),
        'std':cv_res[f'test_{m}'].std()} for m in scoring}
    print(f'F1={resultados_cv[nombre]["f1"]["mean"]:.3f} +/- {resultados_cv[nombre]["f1"]["std"]:.3f}')
joblib.dump(resultados_cv, CHECKPOINTS/'03b_cv_DCSA.pkl')
print('CV completada')

In [ ]:
# Tabla resumen CV
metricas = ['f1','roc_auc','balanced_accuracy','precision','recall']
filas = []
for nombre, res in resultados_cv.items():
    fila = {'Modelo':nombre}
    for m in metricas:
        fila[f'{m}_mean'] = round(res[m]['mean'],4)
        fila[f'{m}_std']  = round(res[m]['std'],4)
    filas.append(fila)
tabla_cv = pd.DataFrame(filas).sort_values('f1_mean', ascending=False)
display(tabla_cv)
tabla_cv.to_csv(OUT_TABLES/'cv_DCSA.csv', index=False)

In [ ]:
# Grafico comparacion modelos
colores_m = ['#2C7BB6','#1A9850','#66C2A5','#F46D43','#D73027','#A50026','#762A83']
fig, axes = plt.subplots(1, 2, figsize=(16,6))
x = np.arange(len(tabla_cv))
w = 0.35

ax = axes[0]
ax.bar(x-w/2, tabla_cv['f1_mean'], w, yerr=tabla_cv['f1_std'],
    color=[colores_m[i%7] for i in range(len(tabla_cv))],
    alpha=0.85, capsize=4, label='F1', edgecolor='white')
ax.bar(x+w/2, tabla_cv['roc_auc_mean'], w, yerr=tabla_cv['roc_auc_std'],
    color=[colores_m[i%7] for i in range(len(tabla_cv))],
    alpha=0.5, capsize=4, label='AUC-ROC', edgecolor='white', hatch='//')
ax.set_xticks(x)
ax.set_xticklabels(tabla_cv['Modelo'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Puntuacion CV 5-fold')
ax.set_title('A. F1 y AUC-ROC por modelo')
ax.legend()
ax.set_ylim(0,1.05)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
for i, (_, row) in enumerate(tabla_cv.iterrows()):
    ax.errorbar(row['precision_mean'], row['recall_mean'],
        xerr=row['precision_std'], yerr=row['recall_std'],
        fmt='o', markersize=10, capsize=4,
        color=colores_m[i%7], label=row['Modelo'])
ax.set_xlabel('Precision')
ax.set_ylabel('Recall')
ax.set_title('B. Precision vs Recall (trade-off)')
ax.legend(fontsize=8, bbox_to_anchor=(1.02,1))
ax.grid(alpha=0.3)

plt.suptitle('Comparacion de modelos — CV 5-fold | DCSA', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIGURES/'08_comparacion_modelos_DCSA.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Entrenamiento final y evaluacion en test set
# El test set se usa UNA SOLA VEZ aqui para la evaluacion definitiva
top3 = tabla_cv.head(3)['Modelo'].tolist()
print(f'Top 3 por F1-CV: {top3}')

entrenados = {}
res_test   = []

for nombre in top3:
    pipe = portafolio[nombre]
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:,1]
    y_pred = pipe.predict(X_test)
    res_test.append({
        'Modelo':    nombre,
        'F1':        round(f1_score(y_test, y_pred),4),
        'AUC_ROC':   round(roc_auc_score(y_test, y_prob),4),
        'Bal_Acc':   round(balanced_accuracy_score(y_test, y_pred),4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0),4),
        'Recall':    round(recall_score(y_test, y_pred),4),
        'Avg_Prec':  round(average_precision_score(y_test, y_prob),4),
    })
    entrenados[nombre] = {'pipeline':pipe,'y_prob':y_prob,'y_pred':y_pred}
    print(f'  {nombre}: F1={res_test[-1]["F1"]:.4f} AUC={res_test[-1]["AUC_ROC"]:.4f}')

tabla_test = pd.DataFrame(res_test).sort_values('F1', ascending=False)
display(tabla_test)
tabla_test.to_csv(OUT_TABLES/'resultados_test_DCSA.csv', index=False)

In [ ]:
# Curvas ROC y Precision-Recall
colores_top = [COLORS['secondary'], COLORS['primary'], COLORS['accent']]
fig, axes = plt.subplots(1, 2, figsize=(14,6))
for i,(nombre,res) in enumerate(entrenados.items()):
    RocCurveDisplay.from_predictions(
        y_test, res['y_prob'], name=nombre,
        color=colores_top[i], ax=axes[0])
    PrecisionRecallDisplay.from_predictions(
        y_test, res['y_prob'], name=nombre,
        color=colores_top[i], ax=axes[1])
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('A. Curvas ROC — Top 3')
axes[0].grid(alpha=0.3)
axes[1].axhline(y_test.mean(), linestyle='--', color='gray',
    label=f'Prevalencia: {y_test.mean():.1%}')
axes[1].set_title('B. Curvas Precision-Recall — Top 3')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.suptitle('Evaluacion en test — DCSA', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIGURES/'09_curvas_ROC_PR_DCSA.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Matrices de confusion
# FN: hospitalizado predicho como no hosp (riesgo alta prematura)
# FP: no hosp predicho como hosp (ocupacion innecesaria de camas)
fig, axes = plt.subplots(1, len(entrenados), figsize=(14,4))
if len(entrenados)==1: axes=[axes]
for i,(nombre,res) in enumerate(entrenados.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, res['y_pred'],
        display_labels=['No hosp.','Hospitalizado'],
        cmap='Blues', ax=axes[i], colorbar=False)
    axes[i].set_title(f'{nombre}\nF1={f1_score(y_test,res["y_pred"]):.3f}',
        fontsize=10, fontweight='bold')
plt.suptitle('Matrices de confusion — test (DCSA)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIGURES/'10_matrices_confusion_DCSA.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Permutation importance — mejor modelo
# Mas confiable que feature_importance_ de arboles:
# no sobreestima variables de alta cardinalidad o correlacionadas
mejor = tabla_test.iloc[0]['Modelo']
pipe_m = entrenados[mejor]['pipeline']
print(f'Calculando permutation importance para: {mejor}')

perm = permutation_importance(
    pipe_m, X_test, y_test,
    n_repeats=20, random_state=RANDOM_STATE,
    scoring='f1', n_jobs=-1)

df_imp = pd.DataFrame({
    'Variable':   FEATURES,
    'Importancia': perm.importances_mean,
    'Std':         perm.importances_std,
}).sort_values('Importancia', ascending=False)
df_imp['Tipo'] = [
    'Signo de alarma' if v in SIGNOS_ALARMA
    else ('Sintoma general' if v in SINTOMAS_GENERALES else 'Demo/Temporal')
    for v in df_imp['Variable']]

display(df_imp.head(15))
df_imp.to_csv(OUT_TABLES/'perm_importance_DCSA.csv', index=False)

In [ ]:
# Grafico de importancia de variables
color_map = {'Signo de alarma':COLORS['con_signos'],
    'Sintoma general':COLORS['sin_signos'],
    'Demo/Temporal':COLORS['accent']}
df_top = df_imp.head(15)

fig, ax = plt.subplots(figsize=(10,7))
colores_b = [color_map[t] for t in df_top['Tipo']]
ax.barh(df_top['Variable'][::-1], df_top['Importancia'][::-1],
    xerr=df_top['Std'][::-1], color=colores_b[::-1],
    capsize=4, alpha=0.85, edgecolor='white')
ax.set_xlabel('Reduccion en F1 (permutation importance)')
ax.set_title(f'Importancia de variables — {mejor} | DCSA')
import matplotlib.patches as mpatches
leyenda = [mpatches.Patch(color=c,label=t) for t,c in color_map.items()]
ax.legend(handles=leyenda, loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIGURES/'11_importancia_DCSA.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Guardar modelos
for nombre, res in entrenados.items():
    nombre_f = nombre.lower().replace(' ','_')
    joblib.dump(res['pipeline'], OUT_MODELS/f'pipeline_DCSA_{nombre_f}.pkl')
joblib.dump({'mejor':mejor,'pipeline':entrenados[mejor]['pipeline'],
    'features':FEATURES,'tabla_test':tabla_test,'df_imp':df_imp},
    CHECKPOINTS/'03b_mejor_DCSA.pkl')
print(f'Mejor modelo guardado: {mejor}')

## Resumen

| Aspecto | Resultado |
|---------|----------|
| Portafolio | 7 modelos evaluados con CV 5-fold |
| Feature mas importante | Caida de plaquetas (consistente con OR ajustados) |
| Desbalance | SMOTE solo en X_train |

**Siguiente:** `04_clustering/04_analisis_no_supervisado.ipynb`